In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.layers import concatenate, Dense
from tensorflow.keras import regularizers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, concatenate, GaussianNoise, Dropout

# Simulation2

In [ ]:
# identity (35), position (100), allography (116)
X_id = pd.read_excel('Target1.xlsx', index_col=0, sheet_name="identity").values.astype(np.float32)
X_pos = pd.read_excel('Target1.xlsx', index_col=0, sheet_name="position").values.astype(np.float32)
X_allog = pd.read_excel('Targetx.xlsx', index_col=0,sheet_name="allog").values.astype(np.float32)
df_sem = pd.read_excel('semantic_vec.xlsx', sheet_name='Sheet1', index_col=None)
Y_sem = df_sem.values[:477, :300].astype(np.float32)


in_id   = Input(shape=(35,),  name='in_id')
tower_id  = Dense(700, activation='sigmoid')(in_id)

in_pos  = Input(shape=(100,), name='in_pos')
#noisy_pos = GaussianNoise(stddev=0.2)(in_pos)                # inject positional noise
tower_pos = Dense(700, activation='sigmoid')(in_pos)

in_allog  = Input(shape=(116,), name='in_allog')
tower_allog = Dense(700, activation='sigmoid')(in_allog)


merged = concatenate([tower_id, tower_pos, tower_allog])
h = Dense(500, activation='sigmoid')(merged)

output = Dense(300, activation='sigmoid', name='sem_output')(h)

model2 = Model(inputs=[in_id, in_pos, in_allog], outputs=output)


model2.compile(optimizer='adam', loss='mse')
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10)
model2.fit([X_id,X_pos,X_allog], Y_sem, batch_size=25, epochs=700, callbacks=[callback])
model2.save('simulation_2_model.h5')

Epoch 1/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0922
Epoch 2/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0020
Epoch 3/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - loss: 0.0018
Epoch 4/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0018
Epoch 5/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0018
Epoch 6/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0018
Epoch 7/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0018
Epoch 8/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.0018
Epoch 9/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.0017
Epoch 10/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0018
Epoch 11/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0018
Epoch 12/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0018
Epoch 13/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0017
Epoch 14/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.0017
Epoch 15/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - lo

# Test the model
We will find the correlation distance between the original word and its different modified forms.

In [ ]:
df_sem = pd.read_excel('semantic_vec.xlsx', sheet_name='Sheet1', index_col=None)
Y_sem = df_sem.values[:477, :300].astype(np.float32)


data_lst= ["TL+Allog1.xlsx", "TL-Allog1.xlsx", "SubL+Allog1.xlsx", "SubL-Allog1.xlsx"]
df_sim2 = pd.DataFrame()
for document in data_lst:
    X_id   = pd.read_excel(document, sheet_name='identity', index_col=0).values.astype(np.float32)
    X_pos  = pd.read_excel(document, sheet_name='position', index_col=0).values.astype(np.float32)
    allog_f = document.replace('1.xlsx','x.xlsx')
    X_allog = pd.read_excel(allog_f, sheet_name='allog', index_col=0).values.astype(np.float32)
    pred = model2.predict([X_id, X_pos, X_allog])
    document = document.replace('1.xlsx','').lower()
    valz = []
    for i in range(len(Y_sem)):
      r = np.corrcoef(pred[i], Y_sem[i])[0,1]
      valz.append(round(1-r,3))


    df_sim2[f'target-{document}']= valz

df_sim2.to_excel('simulation2_correlations_test_0.2_noise_10.xlsx')

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


# Correlation Distance Between words that share the same root

In [ ]:
import itertools

df_words = pd.read_excel('Target1.xlsx', sheet_name='word', index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model2.predict([X_id, X_pos, X_allog])

df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words


# Load roots file
df_roots = pd.read_excel('targets with roots and semantics.xlsx', sheet_name='grouped by roots',     index_col=0)


# Merge so each row has its vector
df_merged = df_roots.merge(df_pred, on='word', how='inner')


# compute all pairwise distances for each root
rows = []
for root, grp in df_merged.groupby('Root'):
    items = list(zip(grp['word'], grp[[f'v{i}' for i in range(300)]].values))
    for (w1, vec1), (w2, vec2) in itertools.combinations(items, 2):
        r = np.corrcoef(vec1, vec2)[0,1]
        dist = 1 - r
        rows.append({'Root': root, 'word1': w1, 'word2': w2, 'distance': dist})

df_pairs = pd.DataFrame(rows)

# Optionally - summary stats per root
summary = df_pairs.groupby('Root')['distance'].agg(
    mean_dist='mean',
    std_dist ='std',
    min_dist ='min',
    max_dist ='max'
).reset_index()


with pd.ExcelWriter('simulation2_root_pairwise_distances.xlsx') as wr:
    df_pairs.to_excel(wr, sheet_name='all_pairs', index=False)
    summary.to_excel(wr, sheet_name='per_root_summary', index=False)



15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


# Correlation Distance Between the word and other 4 conditions

In [ ]:
import numpy as np
import pandas as pd

df_words = pd.read_excel('Target1.xlsx', sheet_name='word', index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model2.predict([X_id, X_pos, X_allog])


df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words


PAIRS_XLSX  = 'targets with roots and semantics.xlsx'
PAIRS_SHEET = '4_conditions'


vec_cols = [c for c in df_pred.columns if c.startswith('v')]
vec_map  = dict(zip(df_pred['word'], df_pred[vec_cols].values))

# read pairs
pairs = pd.read_excel(PAIRS_XLSX, sheet_name=PAIRS_SHEET)

def corr_distance(w1, w2):
    v1 = vec_map.get(w1)
    v2 = vec_map.get(w2)
    if v1 is None or v2 is None:
        return np.nan
    r = np.corrcoef(v1, v2)[0, 1]
    return 1 - r

pairs['distance'] = pairs.apply(lambda r: corr_distance(r['word1'], r['word2']), axis=1)

# keep exactly 4 columns in this order
out = pairs[['word1', 'word2', 'condition', 'distance']]
out.to_excel('simulation2_4_conditions_pairs_with_distance_10.xlsx', index=False)



15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
